<br><br>
<h1><font color="#B30033" size=5>Intelligent Systems - Academic year 2026-2027</font></h1>

<h1><font color="#B30033" size=5>Lab 2: Reinforcement learning</font></h1>

<br>
<div style="text-align: left">
<font color="#4E70BE" size=3>Teaching staff:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Mª Julia Flores Gallego (Julia.Flores@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Antonio Gámez Martín (Jose.Gamez@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Ismael García Varea (Ismael.Garcia@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Roldán Gómez (Jose.Roldan@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Bastián Andrés Troncoso Retamales (Bastian.Troncoso@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Teresa Valero Díaz (Teresa.Valero@uclm.es)</font><br></li>
</ul>
</div>

<div style="text-align: left">
<font color="#4E70BE" size=3>Submitted by:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Full name, e-mail</font><br></li>
  <li><font color="#4E70BE" size=3>Full name, e-mail</font><br></li>
</ul>
</div>

--------------------


## 1. Introduction

In Lab 1 the agent knew everything: the map, the costs and where each action leads. With
that information, finding the best path is planning, and A$^*$ solves it.

This lab takes that information away. **The agent cannot see the plan of the station**: it
can only act and observe where it ends up and what reward it gets. On top of that,
movement is no longer reliable: the chosen action is executed most of the time, but not
always.

Without a model of the world you cannot plan, so you have to **learn from experience**:
try, get it wrong, and keep what works. That is reinforcement learning, and what is
learned is not a path but a **policy**: what to do at each position.

And how do you know whether what was learnt is good? In the second part you will compute the
true **optimal policy** with **value iteration**, which does use the model of the world (the
probability of each transition), and measure how close Q-learning gets to it without one.

By the end you should be able to:

* **Model** a task as a Markov decision problem: states, actions, dynamics and rewards.
* **Implement** tabular Q-learning with an $\varepsilon$-greedy exploration policy.
* **Compute the optimal policy** with value iteration when the model is known, and compare
  with it what is learnt without a model.
* **Tune** $\alpha$, $\gamma$ and the $\varepsilon$ decay schedule, and explain what each
  one does.
* **Design rewards**, and recognise when a badly chosen reward produces an agent that does
  exactly what you did not want.
* **Evaluate a policy**, which is not the same as looking at the Q table.


## 2. Problem description

The station is the same one as in Lab 1, with its levels, bulkheads, expensive areas and
hatches. What changes is what the agent knows and what happens when it acts.

| Element | In this lab |
|---|---|
| State | The position `[level, row, column]` where the agent is |
| Actions | `NORTE`, `ESTE`, `SUR`, `OESTE`, `SUBIR`, `BAJAR` |
| Dynamics | Unknown to the agent, and **stochastic** |
| Reward | Whatever the environment returns at each step |
| Episode | Starts at the origin and ends on reaching the destination or running out of steps |

### 2.1. Movement fails

With probability `p_exito` the chosen action is executed. Otherwise:

* if the action was **horizontal**, the agent **slips** and another horizontal action is
  executed at random;
* if it was `SUBIR` or `BAJAR`, **the hatch jams** and the agent stays where it is.

And if the action that finally gets executed is not possible —there is a bulkhead, or
there is no hatch—, the agent **collides**: it does not move and gets the collision
reward.

Besides, every episode has a **step limit** (`limite_pasos`): if the agent has not reached
the destination when it runs out, the episode is cut off. It is 4 × rows × columns of the
station: 400 steps in `PR2-S-01` and 1,600 in the M instances. Like `p_exito`, it is part of
the dynamics and comes in the instance, in `consulta.dinamica`. It is not a reward, and
cutting an episode off for time is not reaching a final state: it is just stopping
watching.

### 2.2. The rewards

They come in the instance, under `consulta.dinamica.recompensas`:

| Reward | Default value | When |
|---|---|---|
| `objetivo` | +100 | On reaching the destination |
| `paso_por_coste` | −1 | Per unit of cost of the executed move |
| `colision` | −5 | On hitting a bulkhead or a hatch that is not there |

A step along a cost-1 corridor gives −1; along a cost-3 service duct, −3. So the agent
does not only learn to arrive: it learns to arrive cheaply, which is exactly what Lab 1
was optimising.

**These rewards can be changed**, and doing so is part of the lab (section 9).


## 3. Working on the lab

You will work on three instances of the catalogue, in `catalogo/pr2/`:

| Instance | States | Deterministic optimal cost (Lab 1) |
|---|---|---|
| `PR2-S-01` | 87 | 21 |
| `PR2-M-01` | 829 | 44 |
| `PR2-M-02` | 822 | 48 |

That last column is the cost of the best path **if movement never failed**, computed with
the algorithm of Lab 1. It is worth spoiling the ending: your policy will not reach it, and
not because it learns badly. The right reference is the **optimal policy of the problem with
slips**, and you will compute it yourself with value iteration (section 7.4).

### 3.1. The environment

It is provided in `ssii_estacion.entorno`. It has two methods, and that is all you get:

```python
estado = entorno.reset()                       # start a new episode
estado, recompensa, terminado, info = entorno.paso(accion)
```

The environment **does not expose the station**. The attribute holding it starts with an
underscore, and reading it is cheating: if you could look at the map, the optimal policy
would be computed with A$^*$ and there would be no lab. This is checked during marking.

For value iteration, the package gives you, separately, the **model** of the problem,
`ModeloEstacion`: for each state and action, the possible transitions with their
probability and reward. It is exactly what the Q-learning agent does not have, so it is
**only used in value iteration**; using it in Q-learning is also detected during marking.

### 3.2. What you implement

1. `q_learning(entorno, episodios, ...)` — tabular Q-learning with $\varepsilon$-greedy
   exploration. Returns the table `Q` and the **learning curve**.
2. `politica_de(Q)` — the greedy policy with respect to Q: at each state, the action with
   the highest value.
3. `iteracion_valores(modelo, gamma, umbral)` — value iteration on the model. Returns the
   optimal utilities `U` and the convergence history.
4. `politica_desde_valores(modelo, U, gamma)` — the greedy policy with respect to U: the
   optimal one.

The rest —policy evaluation, solution format, validation— is provided by the package, so
that the numbers of every submission are comparable.


## 4. Work plan

There are four lab sessions. A reasonable way to split them:

1. **Session 1**: the environment and Q-learning. Run episodes with random actions, look at
   what `paso` returns and check how often the executed action is not the chosen one.
   Implement Q-learning, get it to learn on `PR2-S-01` and look at its policy (7.1-7.3).
2. **Session 2**: value iteration. Implement it with the model, obtain the optimal policy of
   `PR2-S-01` and compare it with the Q-learning one using `evaluar_politica` and the maps
   (7.4-7.5).
3. **Session 3**: hyperparameters and scale. $\alpha$, $\gamma$, the $\varepsilon$
   decay schedule and the number of episodes on the M instances; learning curves, and what
   share of the optimal reward Q-learning achieves in each case (9.1-9.2).
4. **Session 4**: reward design, analysis questions and report (9.3-9.4).

It always pays to start with `PR2-S-01`: 87 states and a few seconds per experiment.
Debugging Q-learning on an M instance is a wasted afternoon.


## 5. Submission and assessment

The work is done **in pairs** (individually is also accepted; groups of three or more are
not allowed). You submit **this very notebook**, completed, together with the JSON
solution files produced in section 8.

Remember that lab work is assessed with two components:

1. **25 %** — code submission and performance on a battery of automatic tests.
2. **75 %** — lab exam, held in the last week of class.

A **minimum of 4 out of 10** in the weighted average is required to pass or compensate
this part of the course.

If you follow **non-continuous assessment**, you also submit the extension in section 10:
you have more time to solve and hand in the lab. Students under continuous assessment do
not have to submit it.

### 5.1. Automatic marking

There is no single correct result here: two runs with different seeds learn different
policies, and both can be good. What is checked is the **behaviour of the submitted
policy**, not the Q table or the number of episodes:

* that the policy uses valid actions and covers the states it goes through;
* that, when run with `evaluar_politica` and the marking seed, it reaches the destination
  consistently;
* that the numbers you declare in the solution file match the ones obtained by
  re-evaluating your policy;
* that value iteration computes the optimal values: they are compared with the reference;
* that Q-learning has neither looked at the map nor used the model.

Some of the checks are in your hands. From the lab folder:

```bash
pip install pytest
python3 -m pytest PR2/pruebas -q
```

Passing them is the minimum, not the mark: marking adds further tests you do not see.

### 5.2. On the use of generative AI tools

Tabular Q-learning is one of those algorithms an assistant writes from memory, so let us
be explicit: **handing in the work is a declaration that it is your own**, and in the lab
exam —worth 75 % of this part— you will have to work on this same code without help. The
course policy described in unit 0 applies.


## 6. The environment at work

We start by checking that everything loads and seeing what it feels like to act without a
map.

In [ ]:
# The ssii_estacion package sits in the folder above, next to the catalogue.
import sys, os
sys.path.append(os.path.abspath(".."))

import random
import matplotlib.pyplot as plt

from ssii_estacion import solucion_pr2, guardar_solucion
from ssii_estacion.entorno import EntornoEstacion, ModeloEstacion, evaluar_politica
from ssii_estacion.vis import dibujar_politica

print("package loaded successfully")

In [ ]:
# One episode with random actions: this is what the world looks like from inside
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
print(entorno)

rnd = random.Random(0)
estado = entorno.reset()
print("initial state:", estado)

for k in range(8):
    accion = rnd.choice(entorno.acciones)
    estado, recompensa, terminado, info = entorno.paso(accion)
    print(f"  chosen {accion:6} -> executed {str(info['accion_ejecutada']):6} "
          f"| state {estado} | r {recompensa:6.1f} | collision {info['choque']}")
    if terminado:
        break

In [ ]:
# How often is the chosen action really executed?
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=7)
entorno.reset()
iguales = 0
N = 2000
for _ in range(N):
    accion = "ESTE"
    _, _, terminado, info = entorno.paso(accion)
    iguales += (info["accion_ejecutada"] == accion)
    if terminado:
        entorno.reset()
print(f"the chosen action was executed {iguales / N:.1%} of the time "
      f"(p_exito = {entorno.p_exito})")

### 6.1. API summary

```python
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)

entorno.acciones          # ('NORTE', 'ESTE', 'SUR', 'OESTE', 'SUBIR', 'BAJAR')
entorno.p_exito           # probability that the chosen action is the executed one
entorno.limite_pasos      # maximum steps per episode (part of the dynamics)
entorno.recompensas       # dict with objetivo, paso_por_coste and colision

estado = entorno.reset()                                  # -> Pos(level, row, col)
estado, recompensa, terminado, info = entorno.paso(accion)
# info: exito (success), coste (cost), choque (collision), accion_ejecutada, pasos

evaluar_politica(entorno, politica, episodios=200, semilla=12345)
# -> tasa_exito (success rate), recompensa_media, recompensa_desv,
#    pasos_medios (mean steps), coste_medio (mean cost)
```

To experiment with reward design (section 9), the constructor accepts them (and also
`p_exito` and `limite_pasos`, although the lab does not need you to change them):

```python
entorno = EntornoEstacion.desde_instancia(ruta, semilla=1,
                                          recompensas={"paso_por_coste": -2})
```

States are `Pos`, that is, named tuples: they work as dictionary keys, which is all a Q
table needs.

And the model, **only for value iteration** (section 7.4):

```python
modelo = ModeloEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json")

modelo.estados            # all traversable positions
modelo.acciones           # the six actions
modelo.origen, modelo.destino
modelo.es_terminal(s)     # True only at the destination
modelo.transiciones(s, a) # [(probability, s2, reward), ...]; probabilities add up to 1
```

## 7. Implementation

### 7.1. Q-learning

The update rule is the one from unit 5:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\big]$$

And exploration is $\varepsilon$-greedy: with probability $\varepsilon$ a random action,
otherwise the best one according to Q. $\varepsilon$ **decreases** along training: early
on it pays to explore, later to exploit what has been learnt.

Note that `Q` is a dictionary of dictionaries and is **not initialised with every state**:
you do not know them. They are created as they show up.

In [ ]:
def q_learning(entorno, episodios, alpha=0.2, gamma=0.95,
               eps0=1.0, eps_min=0.05, fraccion_decaimiento=0.8,
               semilla=0, ventana=100):
    """Tabular Q-learning with decreasing epsilon-greedy exploration.

    Returns (Q, curva), where curva[i] is the mean reward of the i-th window of
    episodes: that is what you plot to see whether the agent is learning.
    """
    rnd = random.Random(semilla)
    acciones = entorno.acciones
    Q = {}
    recompensas, curva = [], []

    for ep in range(episodios):
        # TODO: compute epsilon for this episode (decreasing down to eps_min)
        eps = None

        s = entorno.reset()
        total = 0.0
        while True:
            # TODO: make sure Q has an entry for s (all actions at 0.0)
            # TODO: pick an action with epsilon-greedy
            accion = None

            # TODO: execute the action in the environment

            # TODO: update Q[s][accion] with the Q-learning rule
            #       careful: the next state must also exist in Q

            # TODO: accumulate the reward and move on to the next state
            break   # <- remove this once the loop is implemented

        recompensas.append(total)
        if (ep + 1) % ventana == 0:
            curva.append(round(sum(recompensas[-ventana:]) / ventana, 2))

    return Q, curva


def politica_de(Q):
    """The greedy policy with respect to Q: at each state, the best-valued action."""
    # TODO: return a dictionary state -> action
    raise NotImplementedError

### 7.2. A quick check

A few thousand episodes are enough on the small instance. Besides the Q table, `q_learning`
returns the **learning curve**: the mean reward per episode over blocks of 100 consecutive
episodes (that is the `ventana=100` parameter). So `curva[0]` is the mean of episodes 1 to
100, `curva[1]` that of episodes 101 to 200, and so on.

If the last values of the curve do not end up clearly above zero, the agent is not learning
to reach the destination: fix it here before moving on to the M instances.

In [ ]:
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
Q, curva = q_learning(entorno, 3000, semilla=1)

print(f"states visited: {len(Q)}")
print(f"curve: {curva[0]} -> {curva[len(curva)//2]} -> {curva[-1]}")

politica = politica_de(Q)
print(evaluar_politica(entorno, politica))

### 7.3. Looking at the policy

A policy is not a path: it is an action for **every** position, because the robot may end up
anywhere when it slips. `dibujar_politica` draws it on the map: an arrow in each position
with the action it chooses (▲ SUBIR, ▼ BAJAR) and, in grey, the positions the agent has
never visited. The map is supplied by the package only for the drawing: your agent still
cannot see it.

Follow the arrows from the origin: you should reach the destination. Look also at the
positions far from that route: do their arrows make sense?

In [ ]:
dibujar_politica(entorno, politica, titulo="PR2-S-01 — learnt policy");

### 7.4. Value iteration: the optimal policy with the model

Q-learning learns without knowing what each action does. If the model is known, the optimal
policy can be **computed**. Value iteration applies the Bellman update of unit 5 to every
state over and over:

$$U_{i+1}(s) \leftarrow \max_a \Big( R(s,a) + \gamma \sum_{s'} T(s,a,s')\, U_i(s') \Big)$$

until the largest change of a sweep, $\delta$, falls below a threshold. The optimal policy is
the greedy one with respect to $U$:

$$\pi^{*}(s) = \arg\max_a \Big( R(s,a) + \gamma \sum_{s'} T(s,a,s')\, U(s') \Big)$$

**$T$ and $R$ come from the model.** `modelo.transiciones(s, a)` returns the list of possible
outcomes of executing $a$ in $s$, each one as a triple `(p, s2, r)`: the probability, the
state reached and the reward received. Therefore:

* $T(s,a,s')$ is the probability `p` of the triple that reaches $s'$;
* $R(s,a)$ is the **expected** reward of executing $a$ in $s$: $\sum p \cdot r$ over the
  triples. It depends on the action, as in the unit: moving along a corridor does not cost
  the same as colliding or going up a hatch.

Three details:

* The destination is the only terminal state. Since the +100 for arriving is already inside
  $R(s,a)$ of the action that arrives, the utility of the destination is 0 (in the algorithm
  of the unit, $U(s) \leftarrow R(s)$ for terminal states, and here $R(\text{destination}) = 0$).
* The problem has an **infinite discounted horizon**: the step limit of the environment is
  not part of the model.
* If two actions tie, either is fine.

In [ ]:
def iteracion_valores(modelo, gamma=0.95, umbral=1e-4, max_iteraciones=10_000):
    """Value iteration on the full model of the station.

    Returns (U, historial): U[s] is the optimal utility of each state (0 at the destination,
    which is terminal) and historial[k] the largest change of U in sweep k (the delta of
    the unit). It stops when that change falls below `umbral` (or after `max_iteraciones`).
    """
    U = {s: 0.0 for s in modelo.estados}
    historial = []

    # TODO: repeat sweeps until the largest change (delta) is below umbral
    #   - at each non-terminal state:
    #       U'(s) = max_a ( R(s,a) + gamma * sum_s' T(s,a,s') * U(s') )
    #     with the triples (p, s2, r) of modelo.transiciones(s, a):
    #       R(s,a) = sum of p * r        and        sum_s' T(s,a,s') * U(s') = sum of p * U[s2]
    #   - store in historial the largest change of the sweep
    raise NotImplementedError


def politica_desde_valores(modelo, U, gamma=0.95):
    """The greedy policy with respect to U: at each non-terminal state, the action that
    maximises R(s,a) + gamma * sum_s' T(s,a,s') * U(s')."""
    # TODO: return a dictionary state -> action (without the destination)
    raise NotImplementedError

### 7.5. The optimal policy versus the learnt one

To compare two policies, **evaluate both with `evaluar_politica`** on the same environment.
Do not compare $U(\text{origin})$ with the mean reward of `evaluar_politica`: $U$ is an
expected **discounted** utility, while `evaluar_politica` measures the total undiscounted
reward of 200 concrete episodes.

The maps help to see where they differ: the first one draws the optimal policy over $U$; the
second, the Q-learning one with the actions that do not match the optimal one in red.

In [ ]:
modelo = ModeloEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json")
U, historial = iteracion_valores(modelo, gamma=0.95)
optima = politica_desde_valores(modelo, U, gamma=0.95)
print(f"{len(historial)} sweeps; last delta {historial[-1]:.1e}; U(origin) = {U[modelo.origen]:.2f}")

print("optimal   :", evaluar_politica(entorno, optima))
print("Q-learning:", evaluar_politica(entorno, politica))

dibujar_politica(modelo, optima, U=U, titulo="PR2-S-01 — optimal policy (value iteration)")
dibujar_politica(entorno, politica, comparar_con=optima, titulo="PR2-S-01 — Q-learning versus the optimal policy");

## 8. Evaluating the policy (not the Q table)

A Q table with good numbers means nothing by itself: what matters is **how the policy that
comes out of it behaves**. `evaluar_politica` runs 200 test episodes with a fixed seed
—the same one for every submission— and returns the success rate, the mean reward with its
standard deviation, the mean number of steps and the mean cost of the route.

That mean cost is comparable with the deterministic optimal cost of Lab 1 and with that of
the optimal policy you computed in 7.4, and that is the interesting conversation of this
lab. Compare them before moving on.

### 8.1. Producing the solution files

This cell writes the JSON files you submit and the marking system checks. Do not modify
it: only change the list of authors and, if you wish, the hyperparameters.

In [ ]:
AUTORES = ["Full name <e-mail@alu.uclm.es>",
           "Full name <e-mail@alu.uclm.es>"]

INSTANCIAS = ["PR2-S-01", "PR2-M-01", "PR2-M-02"]
EPISODIOS = {"PR2-S-01": 3000, "PR2-M-01": 8000, "PR2-M-02": 8000}

HIPER = {"alpha": 0.2, "gamma": 0.95, "eps0": 1.0, "eps_min": 0.05,
         "fraccion_decaimiento": 0.8, "semilla": 1}

import time, os

def entrenar_y_exportar(nombre_instancia, carpeta="soluciones"):
    entorno = EntornoEstacion.desde_instancia(f"../catalogo/pr2/{nombre_instancia}.json",
                                              semilla=HIPER["semilla"])
    t0 = time.perf_counter()
    Q, curva = q_learning(entorno, EPISODIOS[nombre_instancia],
                          alpha=HIPER["alpha"], gamma=HIPER["gamma"],
                          eps0=HIPER["eps0"], eps_min=HIPER["eps_min"],
                          fraccion_decaimiento=HIPER["fraccion_decaimiento"],
                          semilla=HIPER["semilla"])
    entrenamiento_s = time.perf_counter() - t0

    politica = politica_de(Q)
    ev = evaluar_politica(entorno, politica)

    doc = solucion_pr2(
        instancia_id=nombre_instancia, algoritmo="q_learning", autores=AUTORES,
        politica=politica,
        hiperparametros={**HIPER, "episodios": EPISODIOS[nombre_instancia]},
        entrenamiento={"episodios": EPISODIOS[nombre_instancia], "ventana": 100,
                       "curva": curva, "tiempo_s": round(entrenamiento_s, 2),
                       "estados_visitados": len(Q)},
        evaluacion=ev,
    )
    guardar_solucion(doc, os.path.join(carpeta, f"{nombre_instancia}.q_learning.json"))
    return {"instancia": nombre_instancia, "estados": len(Q),
            "tiempo_s": round(entrenamiento_s, 1), **ev}, curva

resultados, curvas = [], {}
for nombre in INSTANCIAS:
    print("training", nombre, "...")
    fila, curva = entrenar_y_exportar(nombre)
    resultados.append(fila)
    curvas[nombre] = curva
print("\ndone: solution files written to ./soluciones/")

## 9. Experimental study

### 9.1. Results

In [ ]:
import pandas as pd

df = pd.DataFrame(resultados)
display(df[["instancia", "estados", "tiempo_s", "tasa_exito",
            "recompensa_media", "recompensa_desv", "pasos_medios", "coste_medio"]])

# And the comparison that gives the lab its point: what is learnt versus the optimum
OPTIMO_PR1 = {"PR2-S-01": 21, "PR2-M-01": 44, "PR2-M-02": 48}
filas = []
for fila in resultados:
    nombre = fila["instancia"]
    ruta = f"../catalogo/pr2/{nombre}.json"
    modelo = ModeloEstacion.desde_instancia(ruta)
    U, _ = iteracion_valores(modelo, gamma=HIPER["gamma"])
    optima = evaluar_politica(EntornoEstacion.desde_instancia(ruta, semilla=HIPER["semilla"]),
                              politica_desde_valores(modelo, U, gamma=HIPER["gamma"]))
    filas.append({"instance": nombre,
                  "Q-learning reward": fila["recompensa_media"],
                  "optimal reward": optima["recompensa_media"],
                  "Q-learning cost": fila["coste_medio"],
                  "optimal cost (VI)": optima["coste_medio"],
                  "optimal cost without slips (Lab 1)": OPTIMO_PR1[nombre]})
display(pd.DataFrame(filas))

### 9.2. Learning curves and policy maps

Plot at least:

* the learning curve of the three instances;
* the effect of $\alpha$ and of $\gamma$ on one of them (several curves on the same axes);
* the effect of the $\varepsilon$ decay schedule: constant versus decreasing;
* and, with `dibujar_politica`, the policy maps proposed after the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for nombre, curva in curvas.items():
    ax.plot([(i + 1) * 100 for i in range(len(curva))], curva, marker="", label=nombre)
ax.axhline(0, color="#adb5bd", lw=.8)
ax.set_xlabel("episodes")
ax.set_ylabel("mean reward (windows of 100)")
ax.set_title("Learning curves")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()

The curve tells how much the agent earns per episode, but not **where** it has learnt. If
you also give `dibujar_politica` the Q table, it paints the learnt value of each position in
the background, $\max_a Q(s, a)$: light areas are those the agent already knows lead to the
destination; in the dark ones, the arrows are little more than noise.

The first cell compares few and many episodes on an M instance. The second compares two
training runs with different seeds: with `comparar_con`, the actions that differ are drawn
in red. Both are the starting point of questions 5, 7 and 8 in section 9.4.

In [ ]:
# The learnt value, with few and with many episodes
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-M-01.json", semilla=1)
for episodios in (1500, 8000):
    Q, _ = q_learning(entorno, episodios, semilla=1)
    dibujar_politica(entorno, politica_de(Q), Q=Q,
                     titulo=f"PR2-M-01 — value and policy after {episodios} episodes");

In [ ]:
# Two training runs, two policies: in how many positions do they differ?
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
Q1, _ = q_learning(entorno, 3000, semilla=1)
Q2, _ = q_learning(entorno, 3000, semilla=2)
p1, p2 = politica_de(Q1), politica_de(Q2)

comunes = [s for s in p1 if s in p2]
print(f"different actions: {sum(p1[s] != p2[s] for s in comunes)} of {len(comunes)}")
print("seed 1:", evaluar_politica(entorno, p1))
print("seed 2:", evaluar_politica(entorno, p2))
dibujar_politica(entorno, p1, comparar_con=p2, titulo="PR2-S-01 — seed 1 versus seed 2");

### 9.3. Reward design

The environment accepts other rewards. Try at least these variants on `PR2-S-01` and
measure the resulting policy with `evaluar_politica`:

1. **No step penalty** (`paso_por_coste: 0`): does it still learn to arrive? Does it
   arrive as cheaply?
2. **Rewarding the step** (`paso_por_coste: +1` and `+3`): look at the success rate and
   the mean number of steps before drawing conclusions. This is textbook *reward hacking*,
   and it is the part that says most in the report.
3. **No collision reward** (`colision: 0`): does the behaviour near bulkheads change?

Produce a table with the five configurations and a paragraph explaining what the agent
learnt in each case. Watch out for the second one: the result also depends on $\gamma$,
and the second cell shows it. Think about what it is worth to go round in circles forever
collecting $r$ per step, compared with the +100 of arriving.

In [ ]:
VARIANTES = {
    "original":           {},
    "free step":          {"paso_por_coste": 0},
    "step rewarded +1":   {"paso_por_coste": +1},
    "step rewarded +3":   {"paso_por_coste": +3},
    "free collision":     {"colision": 0},
}

filas = []
for nombre, cambio in VARIANTES.items():
    entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json",
                                              semilla=1, recompensas=cambio)
    Q, _ = q_learning(entorno, 3000, semilla=1)
    ev = evaluar_politica(entorno, politica_de(Q))
    filas.append({"recompensas": nombre, **ev})

display(pd.DataFrame(filas)[["recompensas", "tasa_exito", "recompensa_media",
                             "pasos_medios", "coste_medio"]])

In [ ]:
# The same reward (+1 per step) with two different gammas
for gamma in (0.95, 0.99):
    entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json",
                                              semilla=1, recompensas={"paso_por_coste": +1})
    Q, _ = q_learning(entorno, 3000, gamma=gamma, semilla=1)
    ev = evaluar_politica(entorno, politica_de(Q))
    print(f"gamma={gamma}: success rate {ev['tasa_exito']:.0%}, "
          f"mean steps {ev['pasos_medios']:.1f}, mean reward {ev['recompensa_media']:.1f}")

### 9.4. Analysis questions

These are the questions your report has to answer, with your data in front of you.

1. **Your policy does not reach the optimal cost of Lab 1.** Give both numbers for the
   three instances and explain why the difference is not a failure of the algorithm. Then
   compare them with the mean cost of the optimal policy computed by value iteration: which
   part of the difference is due to slips, and which part to what Q-learning has not learnt?

2. **Decreasing $\varepsilon$ versus constant $\varepsilon$.** Train with a fixed
   $\varepsilon = 0.1$ and compare the curve with the decreasing schedule. Which learns
   sooner? Which ends up better? Why?

3. **The role of $\gamma$.** Try $\gamma \in \{0.5;\ 0.9;\ 0.95;\ 0.99\}$ on an M
   instance. What happens to the policy with a small $\gamma$? Relate it to the distance
   between origin and destination.

4. **The role of $\alpha$.** With a large $\alpha$ the agent learns fast and forgets fast.
   Check it: give one curve with $\alpha = 0.8$ and another with $\alpha = 0.05$.

5. **Visited states.** Compare the number of states in your Q table with the number of
   traversable positions of the instance. Why do they not match? Is that a problem?
   Use the policy map to see which ones are missing and where they are.

6. **Reward hacking.** When the step is rewarded, the agent stops going to the destination
   as soon as the per-step reward is high enough. Describe exactly what it does and why it
   is perfectly rational given the rewards. Then be precise: going round in circles forever
   collecting $r$ per step is worth $r/(1-\gamma)$, and arriving is worth +100. With
   $\gamma = 0.95$, beyond which $r$ does it pay not to arrive at all? Check your
   prediction with the environment, and explain why with $r = +1$ it is enough to raise
   $\gamma$ to 0.99 for the agent to start circling. (In the environment nothing lasts
   forever: the episode is cut off when `limite_pasos` runs out. Does that change the
   conclusion?) What does this say about specifying objectives for an AI system?

7. **Policy versus Q table.** Two training runs with different seeds give different Q
   tables. Can they give the same policy? And different policies of the same quality?
   Check it with three seeds, `evaluar_politica` and `dibujar_politica(..., comparar_con=...)`.

8. **How the value spreads.** In the value map of section 9.2, why does the high value appear
   only near the destination when there are few episodes? How does it reach the other
   levels? Relate it to the Q-learning update rule and to $\gamma$, and explain what it
   means for the policy in the positions furthest from the destination.

9. **What is learnt versus the optimum.** What fraction of the mean reward of the optimal
   policy does your Q-learning achieve on each instance? In the map with `comparar_con`,
   where are the positions in which the two policies differ: along the usual route or far
   from it? Why do so many differences change the reward so little?

10. **Optimal with slips versus A$^*$.** Can you find in the value iteration map a position
    where the optimal policy does not follow the shortest path of Lab 1? Explain that
    decision with the transitions of the model (what happens if it slips next to a bulkhead
    or a costly area). If you cannot find any, explain why in this station the risk of
    slipping hardly changes the path.

## 10. Extensions

**Non-continuous assessment: this section is compulsory.** You have more time to solve and
hand in the lab, so your submission also includes what is asked here. **Continuous
assessment: it is optional**, and whatever you submit counts towards the code mark.

For each one, export the solution files with the algorithm name given below and add a
table and a paragraph with what you observe.

* **SARSA.** Implement SARSA and compare it with Q-learning on the three instances: curve,
  success rate and mean cost. SARSA learns the policy it actually follows, exploration
  included; Q-learning learns the greedy one. Does it show in an environment where you
  slip? Algorithm: `sarsa`.
* **Optimistic initialisation.** Instead of starting with Q at zero, start with a high
  value. Explain why that explores without needing $\varepsilon$, and measure how long it
  takes to converge compared with $\varepsilon$-greedy. Algorithm: `q_learning_optimista`.

## 11. References

* S. Russell and P. Norvig. *Artificial Intelligence: A Modern Approach*, 4th ed., Pearson,
  2021. Chapter 22.
* R. S. Sutton and A. G. Barto. *Reinforcement Learning: An Introduction*, 2nd ed., MIT
  Press, 2018. Chapters 5 and 6.
* Lecture slides of unit 5.
